# 🗑️ ScrapNet – Waste Classification System
### Local / Jupyter Notebook
**Experiments:** EfficientNet-B0 (no aug) · EfficientNet-B0 (aug) · EfficientNet-B3 (aug)  
Device auto-detected: **CUDA → MPS → CPU**  
Requires `scrapnet_utils.py` in the **same folder** as this notebook.

## Step 1 · Install Dependencies (run once)

In [1]:
# Run this cell once, then you can skip it on subsequent runs
import sys
!{sys.executable} -m pip install -q torch torchvision scikit-learn matplotlib pillow
print("Dependencies ready.")

Dependencies ready.


## Step 2 · Configuration
> **Edit `DATA_DIR`** to point to your dataset root (the folder containing class subfolders).

In [2]:
import sys, os
from pathlib import Path

# ── EDIT THIS ─────────────────────────────────────────────────────────────────
DATA_DIR = '/Users/bhavyabansal/Documents/pbl/scrapnet_updated/master'
# ─────────────────────────────────────────────────────────────────────────────

OUTPUT_DIR  = str(Path.cwd() / 'experiments')   # outputs next to this notebook
EPOCHS      = 15
LR          = 1e-4
SEED        = 42
NUM_WORKERS = 2   # set to 0 if you hit multiprocessing errors on Windows

# Make sure scrapnet_utils is importable from this notebook's directory
sys.path.insert(0, str(Path.cwd()))

from scrapnet_utils import get_device, seed_everything
device = get_device()
seed_everything(SEED)

print(f"DATA_DIR   : {DATA_DIR}")
print(f"OUTPUT_DIR : {OUTPUT_DIR}")

# Quick sanity check on dataset
classes_found = sorted(os.listdir(DATA_DIR))
print(f"Classes ({len(classes_found)}): {classes_found}")

[Device] Using: mps
DATA_DIR   : /Users/bhavyabansal/Documents/pbl/scrapnet_updated/master
OUTPUT_DIR : /Users/bhavyabansal/Documents/pbl/scrapnet_updated/experiments
Classes (8): ['.DS_Store', 'cardboard', 'compost', 'glass', 'metal', 'paper', 'plastic', 'trash']


## Experiment 1 · EfficientNet-B0 — No Augmentation (Baseline)
| Setting | Value |
|---|---|
| Model | EfficientNet-B0 |
| Image size | 224 × 224 |
| Augmentation | None |
| Output | `experiments/b0_no_aug/` |

In [ ]:
from scrapnet_utils import run_experiment

model_b0_noaug, history_b0_noaug, summary_b0_noaug = run_experiment(
    exp_name        = 'b0_no_aug',
    model_name      = 'b0',
    img_size        = 224,
    augment         = False,
    data_dir        = DATA_DIR,
    base_output_dir = OUTPUT_DIR,
    batch_size      = 32,
    epochs          = EPOCHS,
    lr              = LR,
    patience        = 4,
    num_workers     = NUM_WORKERS,
    seed            = SEED,
    device          = device,
)


  EXPERIMENT: b0_no_aug
  Model: EfficientNet-B0 | Augment: False | IMG: 224x224
[Data] Train: 5693 | Val: 1220 | Test: 1221
[Data] Classes: ['cardboard', 'compost', 'glass', 'metal', 'paper', 'plastic', 'trash']
[Model] EfficientNet-B0 | Params: 4,016,515


/opt/anaconda3/lib/python3.13/site-packages/torch/utils/data/dataloader.py:1118: UserWarning: 'pin_memory' argument is set as true but not supported on MPS now, device pinned memory won't be used.
  super().__init__(loader)


  Epoch 01/15 | train loss 1.1698  acc 0.5745 | val loss 0.6565  acc 0.7508


## Experiment 2 · EfficientNet-B0 — With Augmentation
| Setting | Value |
|---|---|
| Model | EfficientNet-B0 |
| Image size | 224 × 224 |
| Augmentation | RandomHorizontalFlip, RandomRotation(15°), ColorJitter |
| Output | `experiments/b0_aug/` |

In [ ]:
from scrapnet_utils import run_experiment

model_b0_aug, history_b0_aug, summary_b0_aug = run_experiment(
    exp_name        = 'b0_aug',
    model_name      = 'b0',
    img_size        = 224,
    augment         = True,
    data_dir        = DATA_DIR,
    base_output_dir = OUTPUT_DIR,
    batch_size      = 32,
    epochs          = EPOCHS,
    lr              = LR,
    patience        = 4,
    num_workers     = NUM_WORKERS,
    seed            = SEED,
    device          = device,
)

## Experiment 3 · EfficientNet-B3 — With Augmentation (Scaled Model)
| Setting | Value |
|---|---|
| Model | EfficientNet-B3 |
| Image size | 300 × 300 |
| Augmentation | RandomHorizontalFlip, RandomRotation(15°), ColorJitter |
| Batch size | 16 (reduced for memory) |
| Output | `experiments/b3_aug/` |

> 💡 **Tip:** If you run out of memory, reduce `batch_size` to 8.

In [ ]:
from scrapnet_utils import run_experiment

model_b3_aug, history_b3_aug, summary_b3_aug = run_experiment(
    exp_name        = 'b3_aug',
    model_name      = 'b3',
    img_size        = 300,
    augment         = True,
    data_dir        = DATA_DIR,
    base_output_dir = OUTPUT_DIR,
    batch_size      = 16,
    epochs          = EPOCHS,
    lr              = LR,
    patience        = 4,
    num_workers     = NUM_WORKERS,
    seed            = SEED,
    device          = device,
)

## Step 3 · Cross-Experiment Comparison
Run this cell after all three experiments complete to generate side-by-side comparisons.

In [ ]:
import json
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path

summaries   = [summary_b0_noaug, summary_b0_aug, summary_b3_aug]
exp_labels  = ['B0\n(no aug)', 'B0\n(aug)', 'B3\n(aug)']
histories   = [history_b0_noaug, history_b0_aug, history_b3_aug]
compare_dir = Path(OUTPUT_DIR) / 'comparison'
compare_dir.mkdir(exist_ok=True)

# ── 1. Bar chart: Best Val Acc & Mean F1 ─────────────────────────────────────
val_accs = [s['best_val_acc'] for s in summaries]
mean_f1s = [s['mean_f1'] for s in summaries]

x = np.arange(len(exp_labels))
width = 0.35
fig, ax = plt.subplots(figsize=(9, 5))
b1 = ax.bar(x - width/2, val_accs, width, label='Best Val Acc', color='steelblue', alpha=0.85)
b2 = ax.bar(x + width/2, mean_f1s,  width, label='Mean F1',     color='darkorange', alpha=0.85)
for bar in b1 + b2:
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.01,
            f'{bar.get_height():.3f}', ha='center', va='bottom', fontsize=10)
ax.set_xticks(x); ax.set_xticklabels(exp_labels)
ax.set_ylim(0, 1.1); ax.set_ylabel('Score')
ax.set_title('Model Comparison – Val Accuracy & Mean F1', fontsize=13)
ax.legend(); ax.grid(axis='y', alpha=0.3)
fig.tight_layout()
fig.savefig(compare_dir / 'model_comparison.png', dpi=300); plt.show()

# ── 2. Per-class F1 grouped bar chart ────────────────────────────────────────
all_classes = list(summaries[0]['per_class_f1'].keys())
f1_matrix   = np.array([[s['per_class_f1'][c] for c in all_classes] for s in summaries])
x = np.arange(len(all_classes))
fig2, ax2 = plt.subplots(figsize=(12, 5))
colors = ['steelblue', 'darkorange', 'seagreen']
for i, (label, color) in enumerate(zip(exp_labels, colors)):
    ax2.bar(x + (i-1)*0.27, f1_matrix[i], width=0.27, label=label.replace('\n', ' '),
            color=color, alpha=0.85)
ax2.set_xticks(x); ax2.set_xticklabels(all_classes, rotation=30, ha='right')
ax2.set_ylim(0, 1.1); ax2.set_ylabel('F1 Score')
ax2.set_title('Per-Class F1 – All Models', fontsize=13)
ax2.legend(); ax2.grid(axis='y', alpha=0.3)
fig2.tight_layout()
fig2.savefig(compare_dir / 'per_class_f1_comparison.png', dpi=300); plt.show()

# ── 3. Val curves overlay ─────────────────────────────────────────────────────
fig3, axes = plt.subplots(1, 2, figsize=(14, 5))
line_labels = ['B0 no aug', 'B0 aug', 'B3 aug']
styles = ['-', '--', ':']
for h, label, ls, col in zip(histories, line_labels, styles, colors):
    ep = range(1, len(h['val_loss']) + 1)
    axes[0].plot(ep, h['val_loss'], linestyle=ls, color=col, linewidth=2, label=label)
    axes[1].plot(ep, h['val_acc'],  linestyle=ls, color=col, linewidth=2, label=label)
for ax, title, ylabel in zip(axes, ['Validation Loss', 'Validation Accuracy'], ['Loss', 'Accuracy']):
    ax.set_title(title); ax.set_xlabel('Epoch'); ax.set_ylabel(ylabel)
    ax.legend(); ax.grid(alpha=0.3)
axes[1].set_ylim(0, 1.05)
fig3.suptitle('Validation Curves – All Models', fontsize=13)
fig3.tight_layout()
fig3.savefig(compare_dir / 'val_curves_overlay.png', dpi=300); plt.show()

# ── 4. Console table ──────────────────────────────────────────────────────────
print(f"{'Experiment':<20} {'Best Val Acc':>12} {'Mean F1':>8}")
print('-' * 44)
for s in summaries:
    print(f"{s['experiment']:<20} {s['best_val_acc']:>12.4f} {s['mean_f1']:>8.4f}")

print(f"\nAll comparison plots saved → {compare_dir}")

## Appendix · Output Folder Structure

In [ ]:
# Print the experiment output tree
import os
from pathlib import Path

exp_root = Path(OUTPUT_DIR)
if exp_root.exists():
    for root, dirs, files in os.walk(exp_root):
        dirs.sort(); files.sort()
        level = root.replace(str(exp_root), '').count(os.sep)
        indent = '  ' * level
        print(f'{indent}{os.path.basename(root)}/')
        for f in sorted(files):
            print(f'{indent}  {f}')
else:
    print("No experiments run yet.")